# Helmholtz PINN — frozen reproduction

This notebook reproduces the fixed twelve-model experiment in a separate checkout. The repository already contains the completed submission. Use a Colab GPU runtime and run cells in order. Additional protected execution requires the trial reviewer’s authorization. Each model has its own progress display; condition metrics are evaluated only after all six models finish.


In [ ]:
from pathlib import Path
from google.colab import drive
import subprocess, sys, datetime, shutil

COMMIT_SHA = "0677b40869fabf50035075e240aea3b98a0c7e05"
drive.mount("/content/drive")
REPO = Path("/content/helmholtz_pinn_reproduction")
BASE = Path("/content/drive/MyDrive/helmholtz_pinn_reproduction")
BASE.mkdir(parents=True, exist_ok=True)
SESSION = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
BACKUP = BASE / SESSION
BACKUP.mkdir(exist_ok=False)
print("Persistent execution records:", BACKUP)


In [ ]:
if not REPO.exists():
    subprocess.run(["git", "clone", "--no-checkout", "https://github.com/Arawal123/helmholtz-pinn-rff-benchmark.git", str(REPO)], check=True)
def git(*args):
    return subprocess.check_output(["git", *args], cwd=REPO, text=True).strip()
if (REPO / "results/raw").exists():
    assert git("rev-parse", "HEAD") == COMMIT_SHA, "Existing evidence belongs to another revision; preserve that checkout."
else:
    subprocess.run(["git", "checkout", "--detach", COMMIT_SHA], cwd=REPO, check=True)
assert git("rev-parse", "HEAD") == COMMIT_SHA
assert not git("status", "--porcelain"), "Scientific checkout must be clean."
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=REPO, check=True)
subprocess.run([sys.executable, "scripts/validate_protocol.py"], cwd=REPO, check=True)
import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU."
print("Scientific revision:", COMMIT_SHA, "| GPU:", torch.cuda.get_device_name(0))
(BACKUP / "code_sha.txt").write_text(COMMIT_SHA + "\n")


In [ ]:
def persist():
    for name in ("results", "figures", "report"):
        source = REPO / name
        if source.exists():
            shutil.copytree(source, BACKUP / name, dirs_exist_ok=True)

def run_stage(*args):
    command = [sys.executable, "-u", "scripts/run_frozen.py", "--config", "configs/frozen.yaml", *args]
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    log = BACKUP / f"console_{stamp}.log"
    print("Command:", " ".join(command), flush=True)
    with log.open("w", encoding="utf-8") as stream:
        with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                stream.write(line)
                stream.flush()
            code = process.wait()
    persist()
    if code:
        raise subprocess.CalledProcessError(code, command)
    print("Stage complete. Evidence saved to", BACKUP, flush=True)

run_stage("--action", "status")


## n=6: primary condition

Complete all six training cells before the condition evaluation cell.


### Vanilla, seed 0


In [ ]:
run_stage("--action", "train", "--n", "6", "--method", "vanilla", "--seed", "0")


### Vanilla, seed 1


In [ ]:
run_stage("--action", "train", "--n", "6", "--method", "vanilla", "--seed", "1")


### Vanilla, seed 2


In [ ]:
run_stage("--action", "train", "--n", "6", "--method", "vanilla", "--seed", "2")


### RFF, seed 0


In [ ]:
run_stage("--action", "train", "--n", "6", "--method", "rff", "--seed", "0")


### RFF, seed 1


In [ ]:
run_stage("--action", "train", "--n", "6", "--method", "rff", "--seed", "1")


### RFF, seed 2


In [ ]:
run_stage("--action", "train", "--n", "6", "--method", "rff", "--seed", "2")


### Evaluate n=6


In [ ]:
run_stage("--action", "evaluate", "--n", "6")


## n=12: stress condition

Complete all six training cells before the condition evaluation cell.


### Vanilla, seed 0


In [ ]:
run_stage("--action", "train", "--n", "12", "--method", "vanilla", "--seed", "0")


### Vanilla, seed 1


In [ ]:
run_stage("--action", "train", "--n", "12", "--method", "vanilla", "--seed", "1")


### Vanilla, seed 2


In [ ]:
run_stage("--action", "train", "--n", "12", "--method", "vanilla", "--seed", "2")


### RFF, seed 0


In [ ]:
run_stage("--action", "train", "--n", "12", "--method", "rff", "--seed", "0")


### RFF, seed 1


In [ ]:
run_stage("--action", "train", "--n", "12", "--method", "rff", "--seed", "1")


### RFF, seed 2


In [ ]:
run_stage("--action", "train", "--n", "12", "--method", "rff", "--seed", "2")


### Evaluate n=12


In [ ]:
run_stage("--action", "evaluate", "--n", "12")


## Aggregate and preserve the completed experiment


In [ ]:
run_stage("--action", "finalize")
run_stage("--action", "status")
archive = shutil.make_archive(str(BACKUP / "reproduction_evidence"), "zip", root_dir=REPO)
print("Completed evidence archive:", archive)
